In [ ]:
import numpy as np
from numpy import ndarray
import matplotlib.pyplot as plt
from tqdm import tqdm


plt.style.use('dark_background')

### Реализация функций активации.

In [ ]:
# ReLU
def relu(x: ndarray) -> ndarray:
    return np.maximum(0, x)

def drelu(act: ndarray, grad: ndarray) -> ndarray:
    return grad*np.where(act>0, 1, 0)

In [ ]:
# ELU
def elu(x: ndarray, alpha: float=1) -> ndarray:
    return np.where(x>0, x, alpha * (np.exp(x) - 1))

def delu(act: ndarray, grad: ndarray, alpha: float=1) -> ndarray:
    return grad*np.where(act>0, 1, act + alpha)

In [ ]:
# Swish
def swish(x: ndarray) -> ndarray:
    sigmoid = (1 / (1 + np.exp(-x)))
    return sigmoid, x * sigmoid

def dswish(
        act: ndarray,
        grad: ndarray,
        sigm: ndarray
) -> ndarray:
    return grad*(act + sigm * (1 - act))

### Данные для обучения сети.

In [ ]:
def generate_images_with_squares(
        num_images: int = 100000,
        contrast: int = 50,
        seed: int = None
) -> tuple[ndarray, ndarray]:
    """Генерирует изображения с вписанными квадратами"""

    rng = np.random.default_rng(seed)
    
    if contrast < 0 or contrast > 100:
        raise ValueError("Параметр 'contrast' должен быть в диапазоне от 0 до 100.")

    c = int(199*contrast/100)

    # Генерация фоновых изображений.
    img = rng.integers(0, 200 - c, [num_images, 64, 64], dtype=np.uint8)

    # Генерация квадратов.
    square = rng.integers(c, 200, [num_images, 15, 15], dtype=np.uint8)

    # Генерация случайных координат для вставки.
    real_coords = rng.integers(20, 44, size=(num_images, 2))

    for i in range(num_images):
        x, y = real_coords[i]        

        # Вставка квадрата в изображение.
        img[i, (y - 7):(y + 8), (x - 7):(x + 8)] = square[i]

    return img, real_coords


In [ ]:
imgs, real_coords = generate_images_with_squares(num_images=1, contrast=50)
plt.imshow(imgs[0], cmap='gray')
plt.plot(real_coords[0, 0], real_coords[0, 1], marker='o', color='green')
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
fig, ax = plt.subplots(3, 5, figsize=(14, 7))
for i, contrast in enumerate(np.linspace(0, 100, 15)):
    imgs, real_coords = generate_images_with_squares(num_images=1, contrast=contrast)
    ax[i//5, i%5].imshow(imgs[0], cmap='gray')
    ax[i//5, i%5].plot(real_coords[0, 0], real_coords[0, 1], marker='o', color='green')
    ax[i//5, i%5].set_title(f"contrast = {float(contrast):.2f}")
    ax[i//5, i%5].set_xticks([])
    ax[i//5, i%5].set_yticks([])

plt.show()

### Наша первая нейронная сеть.

In [ ]:
def normalize(sample: ndarray) -> ndarray:
    s_min = np.min(sample, axis=-1, keepdims=True)
    s_max = np.max(sample, axis=-1, keepdims=True)
    return (sample - s_min) / (s_max - s_min)*2 - 1


def initialization_param(
        size: tuple[int, int],
        mode: str = "randn",
        bound: float = None,
        seed: int = None
) -> ndarray:
    rng = np.random.default_rng(seed)

    if mode == "randn":
        return rng.standard_normal(size=size)
    elif mode == "uniform":
        return rng.uniform(-bound, bound, size=size)
    else:
        raise ValueError(f"Параметр 'mode' принимает два занчения: 'randn' и 'uniform'. Вы передали '{mode}'.")
    

def create_layers(
        in_features: int,
        out_features: int,
        mode: str = "randn",
        seed: int = None
):
    # Первый линейный слой
    W1 = initialization_param((in_features, 128), mode=mode, bound=np.sqrt(6 / in_features), seed=seed)
    b1 = initialization_param((1, 128), mode=mode, bound=np.sqrt(6 / 128), seed=seed)

    # Второй линейный слой
    W2 = initialization_param((128, out_features), mode=mode, bound=np.sqrt(6 / 128), seed=seed)
    b2 = initialization_param((1, out_features), mode=mode, bound=np.sqrt(6 / 2), seed=seed)

    return W1, b1, W2, b2


def train(
        data: tuple[ndarray, ndarray],
        in_features: int,
        out_features: int,
        init_mode: str = "randn",
        batch_size: int = 16,
        epochs: int = 1,
        lr: float = 0.001,
        shuffle: bool = False,
        norm = False,
        seed = None        
) -> tuple:
    # Создаю параметры нейронной сети.
    W1, b1, W2, b2 = create_layers(in_features, out_features, mode=init_mode, seed=seed)

    imgs, real_coords = data

    data_size = imgs.shape[0]
    num_batches = data_size // batch_size
    total = num_batches * batch_size

    print(f"Размер батча = {batch_size}")
    print(f"Кол-во целых батчей = {num_batches}", end="\n\n")

    # Цикл тренировки.
    for epoch in range(epochs):

        # Если нужно, то перемешиваем данные.
        if shuffle:
            rng = np.random.default_rng(seed + epoch if seed else None)
            # Формируем индексы в рандомном порядке.
            idx = rng.permutation(data_size)
            # Перемешиваем данные.
            imgs = imgs[idx]
            real_coords = real_coords[idx]

        # Разбиваю данные на батчи.
        imgs_batches = imgs[:total].reshape(num_batches, batch_size, in_features)
        coords_batches = real_coords[:total].reshape(num_batches, batch_size, out_features)

        # Нормирую данные при необходимости.
        if norm:
            imgs_batches = normalize(imgs_batches)

        train_loop = tqdm(
            zip(imgs_batches, coords_batches),
            total=num_batches,
            leave=False,
            desc=f"Epoch [{epoch+1}/{epochs}]"
        )

        for batch_idx, (img_batch, coord_batch) in enumerate(train_loop):

            # Прямой проход \ Forward.
            out1 = img_batch@W1 + b1   # size(bs, 128)
            act1 = relu(out1)          # size(bs, 128)

            pred = act1@W2 + b2        # size(bs, 2)

            # Подсчёт функции потерь.
            E = coord_batch - pred
            R = E**2
            s = np.sum(R)
            MSE = s / R.size

            # Обратный проход \ Backward.
            dMSE = 1
            ds = dMSE * 1/R.size
            dR = ds * np.ones_like(R)
            dE = dR * 2 * E
            dpred = -dE

            db2 = np.sum(dpred, axis=0, keepdims=True)
            dW2 = act1.T @ dpred

            dact1 = dpred @ W2.T

            dout1 = drelu(act1, dact1)

            db1 = np.sum(dout1, axis=0, keepdims=True)
            dW1 = img_batch.T @ dout1

            # Шаг оптимизации согластно алгоритму градиентного спуска.
            W1 -= lr*dW1
            b1 -= lr*db1
            W2 -= lr*dW2
            b2 -= lr*db2

            if batch_idx % 10 == 0:
                train_loop.set_description(f"Epoch [{epoch+1}/{epochs}], MSE = {MSE:.4f}")

            if batch_idx % 100 == 0:
                log_prediction(imgs[batch_idx], real_coords[batch_idx], W1, b1, W2, b2, norm)

    return W1, b1, W2, b2

def log_prediction(
    img: ndarray,
    real_coord: ndarray,
    W1: ndarray,
    b1: ndarray,
    W2: ndarray,
    b2: ndarray,
    norm: bool,
) -> None:
    """Логируем предсказание на одном примере."""
    img = img.reshape(1, -1)
    
    if norm:
        img = normalize(img)
    
    out1 = img @ W1 + b1
    act1 = relu(out1)
    pred = act1 @ W2 + b2
    
    print(f"real_coord = {real_coord}")
    print(f"pred = {pred[0].round(3)}\n")



In [ ]:
# Генерация данных.
data_size = 10000
contrast = 30
seed = 1961

data = generate_images_with_squares(
    num_images=data_size,
    contrast=contrast
)


In [ ]:
W1, b1, W2, b2 = train(
    data,
    in_features = 64*64,
    out_features = 2,
    init_mode = "uniform",
    batch_size = 16,
    epochs = 1,
    lr = 0.001,
    shuffle = True,
    norm = True,
    seed = seed
)

In [ ]:
# Проверяю результаты после обучения сети.
contrast = 30
row =2
col = 5

# Генерирую данные.
imgs, real_coords = generate_images_with_squares(num_images=row*col, contrast=contrast)

# Нормирую даные
imgs = normalize(imgs.reshape(row*col, -1))

# Передаю данные в сеть и получаю от неё ответ.
out1 = imgs@W1 + b1
act1 = relu(out1)
preds = act1@W2 + b2

fig, ax = plt.subplots(row, col, figsize=(14, 7))
for i, (img, pred, r_coord) in enumerate(zip(imgs, preds, real_coords)):
    ax[i//5, i%5].imshow(img.reshape(64, 64), cmap='gray')
    ax[i//5, i%5].plot(pred[0], pred[1], marker='o', color='red')
    ax[i//5, i%5].plot(r_coord[0], r_coord[1], marker='o', color='green')
    ax[i//5, i%5].set_title(f"contrast = {float(contrast):.2f}")
    ax[i//5, i%5].set_xticks([])
    ax[i//5, i%5].set_yticks([])

plt.show()